# Recall Assistance Fee — Extract

`recall_assistance_fee = PRN billed quantity  x  per-unit recall charge from the contract clause`

Quantity basis is **PRN billed quantity** (`pos.pos_invoice.invoice_type_id = 6`,
quantity from `pos.pos_invoice_product_details`). The per-unit charge comes from
the CLM contract clause `recall_charges_rs_v1`, gated on `recall_assist_enable_v1`
and on the contract being **in force on the PRN date**.

## Why the previous version returned fewer records

| # | Bug | Effect |
|---|---|---|
| 1 | `upper(trim(cti.legal_name)) = 'BLINK COMMERCE PRIVATE LIMITED'` sat in `WHERE` after two `LEFT JOIN`s | Silently demoted both joins to `INNER`. Any PRN whose vendor had no active city mapping / tax-info row was dropped. Exact match also missed punctuation variants (`BLINK COMMERCE PVT. LTD.`) that the already-computed-but-unused `normalized_name` would have caught. **Largest single loss.** |
| 2 | `manufacturer_ids[1]` | A contract mapped to N manufacturers priced only 1 of them. The other N-1 got no fee. |
| 3 | `cross join unnest` over `$.values` | Emits nothing for an empty array, dropping the clause row. `$.Values` (capital V) also occurs in the wild. |
| 4 | `Recall_Assistance_Fees = 'Yes'` | Case-sensitive; `'YES'` / `'true'` dropped the contract. |
| 5 | `rtv_elg = 1` gate, with `scope_type = 2` never selected | Brand-scoped return policies were unreachable dead code, so brand-scoped items failed the gate. |

And these corrupted the **fee** rather than the row count:

| # | Bug | Effect |
|---|---|---|
| 6 | `SELECT DISTINCT variant_id, manufacturer_id FROM po.purchase_order_items` | A variant bought from >1 manufacturer produced >1 row, fanning out and **multiplying `prn_qty`**. |
| 7 | `vms_vendor_city_mapping` joined on `vendor_id` | One row per vendor x city; a vendor in 20 cities multiplied `prn_qty` by 20. |
| 8 | `recall` CTE emitted >1 row per manufacturer | Two contracts with different charges double-counted the quantity. |
| 9 | `min(answer_values)` on a varchar | Lexical sort, so `'100'` beats `'9'` — wrong charge picked. |
| 10 | `cast(recall_charges as double)` | Hard cast; one `'1,500'` or `'Rs 50'` answer failed the whole batch. |
| 11 | `CAST(SUM(...) AS BIGINT)` on the fee | Truncated paise. |
| 12 | `rpc.product_product` joined on `variant_id` with no `max(id)` dedup | Fanned out `prn_qty` again. |
| 13 | No contract validity window | Charges from expired / not-yet-effective contracts were applied. |

## Output

`OUT_DIR/recall_assistance_fee.parquet` — one row per (duration x item x manufacturer).
The filename deliberately contains *recall*, *assistance* and *fee* so the Streamlit
app in `README.md` picks it up under the "Recall assistance fee" tab. Copy it into
`dashboard/data/`.

> **Not yet executed against Trino.** Every column path is taken from queries already
> running in this repo, but the joins below are restructured, so run the probe cells
> first and confirm the reconciliation cell at the end before trusting the numbers.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
import os
from datetime import date

OUT_DIR = os.path.expanduser('~/contracts_extract')
os.makedirs(OUT_DIR, exist_ok=True)

START_DATE = '2025-01-01'
END_DATE   = date.today().isoformat()

# 'day' | 'week' | 'month' — grain of the `duration` column.
DAY_WEEK_MONTH = 'month'

# PRN scan is batched so a wide window does not time out.
BATCH_DAYS = 30

# Entity filter. Matched against regexp_replace(upper(trim(legal_name)),'[^a-zA-Z0-9]','')
# so punctuation and spacing variants all collapse to the same token.
ENTITY_TOKEN = 'BLINKCOMMERCE'

FEE_PATH = os.path.join(OUT_DIR, 'recall_assistance_fee.parquet')

print(f'OUT_DIR    = {OUT_DIR}')
print(f'window     = {START_DATE} -> {END_DATE}')
print(f'grain      = {DAY_WEEK_MONTH}   batch = {BATCH_DAYS}d')
print(f'entity     = %{ENTITY_TOKEN}%')
print(f'FEE_PATH   = {FEE_PATH}')

In [ ]:
# =============================================================================
# Trino connection + helpers
# =============================================================================
import re, time
import datetime as dt
import pandas as pd
import pencilbox as pb

conn = pb.get_connection('[Warehouse] Trino')


def log(msg, label=None):
    tag = f'[{label}] ' if label else ''
    print(f"[{dt.datetime.now().strftime('%H:%M:%S')}] {tag}{msg}")


def render(query, **params):
    '''Substitute Redash-style {{placeholders}}; refuse to run a half-rendered query.'''
    out = query
    for key, val in params.items():
        out = re.sub(r'\{\{\s*' + re.escape(key) + r'\s*\}\}', str(val), out)
    leftover = re.findall(r'\{\{.*?\}\}', out)
    if leftover:
        raise ValueError(f'Unsubstituted placeholders remain: {sorted(set(leftover))}')
    return out


def sql(query, label=None, **params):
    if params:
        query = render(query, **params)
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=conn)
    log(f'{time.time()-t0:6.1f}s  {len(df):>10,} rows', label=label)
    return df


def describe(schema, table):
    df = pd.read_sql_query(sql=f'''SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table}'
        ORDER BY ordinal_position''', con=conn)
    print(f'\n-- {schema}.{table}  ({len(df)} cols) --')
    return df


def date_windows(start_str, end_str, step_days):
    '''Inclusive [start, end] split into <=step_days chunks.'''
    cur = dt.date.fromisoformat(start_str)
    end = dt.date.fromisoformat(end_str)
    while cur <= end:
        stop = min(cur + dt.timedelta(days=step_days - 1), end)
        yield cur.isoformat(), stop.isoformat()
        cur = stop + dt.timedelta(days=1)


def ckpt_ok(path):
    return os.path.exists(path) and os.path.getsize(path) > 0


# NOTE: pandas hands the SQL string to a pyformat-paramstyle driver, so a bare `%`
# is read as a format spec and raises. Every LIKE wildcard below is doubled (`%%`).

## Probes

Run these before the extract. Probe 2 and Probe 3 are the ones that quantify the
record loss described above — they are the evidence, not decoration.

In [ ]:
# =============================================================================
# PROBE 1 — every source table resolves, with the columns this query assumes
# =============================================================================
PROBE_TABLES = [
    ('pos', 'pos_invoice'),                     # PRN header  (invoice_type_id = 6)
    ('pos', 'pos_invoice_product_details'),     # PRN line    (quantity)
    ('vms', 'vms_vendor_city_mapping'),         # vendor -> city
    ('vms', 'vms_vendor_city_tax_info'),        # city   -> legal_name
    ('clm', 'clm_contract'),                    # contract header + validity dates
    ('clm', 'clm_contract_section'),
    ('clm', 'clm_section_question_answer'),     # the clause answers
    ('clm', 'clm_contract_mapping'),            # contract -> manufacturer
    ('rpc', 'product_product'),                 # variant -> item -> brand
    ('rpc', 'product_brand'),
    ('rpc', 'product_manufacturer'),
    ('rpc', 'return_policy'),                   # RTV eligibility flag
    ('rpc', 'return_policy_bucket_attribute'),
    ('ba_etls', 'item_details'),                # manufacturer fallback
]

probe = {}
for schema, tbl in PROBE_TABLES:
    try:
        df = describe(schema, tbl)
        print(df.to_string(index=False))
        probe[(schema, tbl)] = df
    except Exception as e:
        print(f'FAILED  {schema}.{tbl}: {e}')
        probe[(schema, tbl)] = None

missing = [k for k, v in probe.items() if v is None or len(v) == 0]
if missing:
    raise RuntimeError(f'HALT - tables missing/empty: {missing}')
print(f'\nOK - all {len(PROBE_TABLES)} source tables resolved')

In [ ]:
# =============================================================================
# PROBE 2 — quantify bug #1 and bug #7 (the entity filter)
#
# Left column  : what the old exact-match `= 'BLINK COMMERCE PRIVATE LIMITED'` keeps.
# Right column : what the normalized `LIKE '%%BLINKCOMMERCE%%'` keeps.
# The gap is records the old query silently discarded.
# =============================================================================
entity_variants = sql('''
    SELECT
        legal_name,
        regexp_replace(upper(trim(legal_name)), '[^a-zA-Z0-9]', '') AS normalized_name,
        COUNT(*) AS n_tax_rows
    FROM vms.vms_vendor_city_tax_info
    WHERE active = 1
      AND lake_active_record = TRUE
      AND regexp_replace(upper(trim(legal_name)), '[^a-zA-Z0-9]', '') LIKE '%%{{token}}%%'
    GROUP BY 1, 2
    ORDER BY n_tax_rows DESC
''', label='entity_variants', token=ENTITY_TOKEN)

print(entity_variants.to_string(index=False))

exact = entity_variants[
    entity_variants['legal_name'].str.upper().str.strip() == 'BLINK COMMERCE PRIVATE LIMITED']
print(f"\nlegal_name spellings matching the normalized token : {len(entity_variants)}")
print(f"...of which the old exact-match predicate accepted : {len(exact)}")
if len(entity_variants) > len(exact):
    print('\n-> The spellings listed above but absent from the exact match were dropped')
    print('   by the previous query. This is bug #1.')

# Bug #7: does a vendor map to more than one city? Each extra city was an extra
# copy of every PRN line for that vendor.
fanout = sql('''
    SELECT n_cities, COUNT(*) AS n_vendors
    FROM (
        SELECT vendor_id, COUNT(*) AS n_cities
        FROM vms.vms_vendor_city_mapping
        WHERE active = 1 AND lake_active_record = TRUE
        GROUP BY 1
    )
    GROUP BY 1 ORDER BY 1
''', label='vendor_city_fanout')
print('\nvendor -> city mapping cardinality (n_cities = the old qty multiplier):')
print(fanout.to_string(index=False))

In [ ]:
# =============================================================================
# PROBE 3 — quantify bug #2, #4, #9, #10 (the contract clause)
# =============================================================================
# 3a. How many recall contracts map to more than one manufacturer? Under the old
#     `manufacturer_ids[1]`, every manufacturer past the first got no fee at all.
multi_mfr = sql('''
    SELECT n_manufacturers, COUNT(*) AS n_contracts
    FROM (
        SELECT cm.contract_id, COUNT(DISTINCT cm.mapped_id) AS n_manufacturers
        FROM clm.clm_contract_mapping cm
        WHERE cm.is_deleted = 0 AND cm.mapping_type = 'MANUFACTURER'
        GROUP BY 1
    )
    GROUP BY 1 ORDER BY 1
''', label='multi_mfr')
print('manufacturers per contract (>1 means the old query under-priced it):')
print(multi_mfr.to_string(index=False))

# 3b. Raw clause answers, unparsed. Check the enable flag's casing (bug #4) and
#     whether any charge string would have failed a hard CAST (bug #10).
clause_raw = sql('''
    SELECT
        qa.question_id,
        json_extract_scalar(json_parse(qa.answer), '$.value') AS answer_value,
        COUNT(*) AS n
    FROM clm.clm_contract c
    JOIN clm.clm_contract_section cs
      ON cs.contract_id = c.id AND cs.is_deleted = 0
    JOIN clm.clm_section_question_answer qa
      ON qa.contract_section_id = cs.id AND qa.is_deleted = 0
    WHERE c.is_deleted = 0
      AND c.state NOT IN ('ARCHIVED', 'REJECTED')
      AND qa.question_id IN ('recall_assist_enable_v1', 'recall_charges_rs_v1')
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
''', label='clause_raw')

for qid, grp in clause_raw.groupby('question_id'):
    print(f'\n-- {qid} --')
    print(grp[['answer_value', 'n']].head(30).to_string(index=False))

enable = clause_raw[clause_raw.question_id == 'recall_assist_enable_v1']
non_yes = enable[(enable.answer_value.notna()) & (enable.answer_value != 'Yes')]
if len(non_yes):
    print("\n-> Enable-flag spellings the old `= 'Yes'` predicate dropped (bug #4):")
    print(non_yes.to_string(index=False))

charges = clause_raw[clause_raw.question_id == 'recall_charges_rs_v1'].dropna(subset=['answer_value'])
bad = charges[~charges.answer_value.str.fullmatch(r'\d+(\.\d+)?', na=False)]
if len(bad):
    print('\n-> Charge strings that would fail a hard CAST AS DOUBLE (bug #10):')
    print(bad.to_string(index=False))

# 3c. Bug #9 in one line: min() over these strings is a lexical sort.
vals = sorted(charges.answer_value.dropna().unique())[:12]
print(f'\nlexical min() over charge strings picks {vals[0]!r} out of {vals}')

In [ ]:
# =============================================================================
# PROBE 4 — quantify bug #6 and #12 (variant -> manufacturer fan-out)
# =============================================================================
po_fanout = sql('''
    SELECT n_mfr, COUNT(*) AS n_variants
    FROM (
        SELECT variant_id, COUNT(DISTINCT manufacturer_id) AS n_mfr
        FROM po.purchase_order_items
        GROUP BY 1
    )
    GROUP BY 1 ORDER BY 1
''', label='po_variant_fanout')
print('distinct manufacturers per variant in po.purchase_order_items')
print('(the old `SELECT DISTINCT variant_id, manufacturer_id` multiplier — bug #6):')
print(po_fanout.to_string(index=False))

pp_fanout = sql('''
    SELECT n_rows, COUNT(*) AS n_variants
    FROM (
        SELECT variant_id, COUNT(*) AS n_rows
        FROM rpc.product_product
        WHERE lake_active_record = TRUE
        GROUP BY 1
    )
    GROUP BY 1 ORDER BY 1 DESC LIMIT 20
''', label='product_product_fanout')
print('\nrows per variant in rpc.product_product with no max(id) dedup (bug #12):')
print(pp_fanout.to_string(index=False))

## Extract

One query. The structural fixes, in the order they appear below:

- **`vendor_entity`** — collapses `vendor_city_mapping x tax_info` to **one row per
  `vendor_id`** carrying an `is_bcpl` flag, matched on the normalized name. Kills the
  city fan-out (#7) and the punctuation misses (#1). The entity restriction is now an
  explicit `INNER JOIN` on that flag, so it cannot silently demote another join.
- **`prod`** — `variant -> item -> brand -> manufacturer`, deduped by `max(id)` per
  variant (#12), and `LEFT JOIN`ing brand/manufacturer so an item with no brand keeps
  its quantity instead of vanishing.
- **`clause_mfr`** — `UNION` of the manufacturers named inside the answer JSON and,
  for contracts where the JSON names none, **every** mapped manufacturer. Replaces
  `manufacturer_ids[1]` (#2).
- **`clause_base`** — `LEFT JOIN UNNEST ... ON TRUE` handling both `$.values` and
  `$.Values` (#3); enable flag compared case-insensitively (#4); charge parsed with
  `regexp_extract` + `TRY_CAST` so `'Rs 1,500'` yields `1500.0` and junk yields `NULL`
  instead of killing the query (#10); `MAX` over the parsed `DOUBLE`, not the varchar (#9).
- **`priced`** — joins the clause on manufacturer **and** on `prn_date` falling inside
  `[effective_date, end_date]` (#13), then `ROW_NUMBER()` picks the single most recently
  updated contract in force, so overlapping contracts cannot double-count (#8).
- **`rtv`** — all three scope types via `UNION ALL`, so brand-scoped policies
  (`scope_type = 2`) are finally reachable (#5). Emitted as a **flag, not a filter** —
  a PRN is itself the return, so gating on RTV eligibility would drop real returns.
  Filter on `rtv_eligible` downstream if you want the old behaviour.

In [ ]:
RECALL_FEE_SQL = '''
WITH
-- ── entity: one row per vendor, no city fan-out (fixes #1, #7) ───────────────
tax_info AS (
    SELECT
        vendor_city_id,
        legal_name,
        regexp_replace(upper(trim(legal_name)), '[^a-zA-Z0-9]', '') AS normalized_name,
        ROW_NUMBER() OVER (PARTITION BY vendor_city_id ORDER BY id DESC) AS rn
    FROM vms.vms_vendor_city_tax_info
    WHERE active = 1 AND lake_active_record = TRUE
),
vendor_entity AS (
    SELECT
        cm.vendor_id,
        MAX(ti.legal_name) AS legal_name,
        MAX(CASE WHEN ti.normalized_name LIKE '%%{{entity_token}}%%' THEN 1 ELSE 0 END) AS is_bcpl
    FROM vms.vms_vendor_city_mapping cm
    INNER JOIN tax_info ti ON ti.vendor_city_id = cm.id AND ti.rn = 1
    WHERE cm.active = 1 AND cm.lake_active_record = TRUE
    GROUP BY cm.vendor_id
),

-- ── product: deduped variant -> item -> manufacturer (fixes #12) ────────────
prod AS (
    SELECT
        pp.variant_id,
        pp.item_id,
        pp.name                  AS item_name,
        pp.brand_id,
        CAST(pm.id AS VARCHAR)   AS manufacturer_id,
        pm.name                  AS manufacturer
    FROM rpc.product_product pp
    INNER JOIN (
        SELECT variant_id, MAX(id) AS max_id
        FROM rpc.product_product
        WHERE approved = 1 AND active = 1 AND lake_active_record = TRUE
        GROUP BY 1
    ) a ON a.variant_id = pp.variant_id AND a.max_id = pp.id
    LEFT JOIN rpc.product_brand b
        ON b.id = pp.brand_id AND b.active = 1 AND b.lake_active_record = TRUE
    LEFT JOIN rpc.product_manufacturer pm
        ON pm.id = b.manufacturer_id AND pm.active = 1 AND pm.lake_active_record = TRUE
),
item_mfr AS (
    SELECT
        variant_id,
        CAST(MAX(manufacturer_id) AS VARCHAR) AS manufacturer_id,
        MAX(manufacturer)                     AS manufacturer
    FROM ba_etls.item_details
    GROUP BY 1
),

-- ── RTV eligibility: all three scope types (fixes #5) ───────────────────────
item_scope AS (
    SELECT pp.item_id, pp.brand_id, b.manufacturer_id
    FROM rpc.product_product pp
    INNER JOIN (
        SELECT item_id, MAX(id) AS max_id
        FROM rpc.product_product
        WHERE approved = 1 AND active = 1 AND lake_active_record = TRUE
        GROUP BY 1
    ) a ON a.item_id = pp.item_id AND a.max_id = pp.id
    LEFT JOIN rpc.product_brand b
        ON b.id = pp.brand_id AND b.active = 1 AND b.lake_active_record = TRUE
),
rtv_src AS (
    -- scope_type 1 = item
    SELECT s.item_id, rr.bucket_name AS bucket
    FROM rpc.return_policy r
    INNER JOIN rpc.return_policy_bucket_attribute rr ON rr.policy_id = r.id AND rr.active = TRUE
    INNER JOIN item_scope s ON s.item_id = r.reference_id
    WHERE r.active = TRUE AND r.scope_type = 1
    UNION ALL
    -- scope_type 2 = brand  (unreachable in the previous version)
    SELECT s.item_id, rr.bucket_name
    FROM rpc.return_policy r
    INNER JOIN rpc.return_policy_bucket_attribute rr ON rr.policy_id = r.id AND rr.active = TRUE
    INNER JOIN item_scope s ON s.brand_id = r.reference_id
    WHERE r.active = TRUE AND r.scope_type = 2
    UNION ALL
    -- scope_type 3 = manufacturer
    SELECT s.item_id, rr.bucket_name
    FROM rpc.return_policy r
    INNER JOIN rpc.return_policy_bucket_attribute rr ON rr.policy_id = r.id AND rr.active = TRUE
    INNER JOIN item_scope s ON s.manufacturer_id = r.reference_id
    WHERE r.active = TRUE AND r.scope_type = 3
),
rtv AS (
    SELECT
        item_id,
        MAX(CASE WHEN bucket = 2 THEN 1 ELSE 0 END) AS dmg_rtv_flag,
        MAX(CASE WHEN bucket = 3 THEN 1 ELSE 0 END) AS nte_rtv_flag
    FROM rtv_src
    GROUP BY 1
),

-- ── contract clause ─────────────────────────────────────────────────────────
raw AS (
    SELECT
        c.id                  AS contract_id,
        c.state               AS contract_state,
        DATE(c.start_date)    AS effective_date,
        DATE(c.end_date)      AS end_date,
        c.updated_at          AS contract_updated_at,
        qa.question_id,
        TRY(json_parse(qa.answer)) AS answer
    FROM clm.clm_contract c
    INNER JOIN clm.clm_contract_section cs
        ON cs.contract_id = c.id AND cs.is_deleted = 0
    INNER JOIN clm.clm_section_question_answer qa
        ON qa.contract_section_id = cs.id AND qa.is_deleted = 0
    WHERE c.is_deleted = 0
      AND c.state NOT IN ('ARCHIVED', 'REJECTED')
      AND qa.question_id IN ('recall_assist_enable_v1', 'recall_charges_rs_v1')
),
format_ AS (
    SELECT
        r.contract_id, r.contract_state, r.effective_date, r.end_date,
        r.contract_updated_at, r.question_id,
        json_extract_scalar(x, '$.Manufacturer.id')   AS json_mfr_id,
        json_extract_scalar(x, '$.Manufacturer.name') AS json_mfr_name,
        json_extract_scalar(x, '$.value')             AS answer_values
    FROM raw r
    LEFT JOIN UNNEST (
        CASE
            WHEN json_extract(r.answer, '$.values') IS NOT NULL
                THEN CAST(json_extract(r.answer, '$.values') AS ARRAY(JSON))
            WHEN json_extract(r.answer, '$.Values') IS NOT NULL
                THEN CAST(json_extract(r.answer, '$.Values') AS ARRAY(JSON))
            ELSE ARRAY[r.answer]
        END
    ) AS t(x) ON TRUE
),
-- every manufacturer the clause applies to, not just element [1] (fixes #2)
ans_mfr AS (
    SELECT DISTINCT contract_id, json_mfr_id AS manufacturer_id, json_mfr_name AS manufacturer
    FROM format_
    WHERE json_mfr_id IS NOT NULL
),
clause_mfr AS (
    SELECT contract_id, manufacturer_id, manufacturer FROM ans_mfr
    UNION
    SELECT DISTINCT
        cm.contract_id,
        CAST(cm.mapped_id AS VARCHAR) AS manufacturer_id,
        cm.mapped_value               AS manufacturer
    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
      AND cm.mapping_type = 'MANUFACTURER'
      AND cm.contract_id NOT IN (SELECT contract_id FROM ans_mfr)
),
clause_base AS (
    SELECT
        contract_id, contract_state, effective_date, end_date, contract_updated_at,
        MAX(CASE WHEN question_id = 'recall_assist_enable_v1'
                 THEN upper(trim(answer_values)) END) AS recall_enabled,
        -- strip thousands separators, take the first number, never hard-cast (#9, #10)
        MAX(CASE WHEN question_id = 'recall_charges_rs_v1'
                 THEN TRY_CAST(regexp_extract(replace(answer_values, ',', ''),
                                              '[0-9]+(?:\.[0-9]+)?') AS DOUBLE) END) AS recall_charge_rs
    FROM format_
    GROUP BY 1, 2, 3, 4, 5
),
clause AS (
    SELECT
        cb.contract_id, cb.contract_state, cb.effective_date, cb.end_date,
        cb.contract_updated_at, cb.recall_charge_rs,
        cmf.manufacturer_id, cmf.manufacturer
    FROM clause_base cb
    INNER JOIN clause_mfr cmf ON cmf.contract_id = cb.contract_id
    WHERE cb.recall_enabled IN ('YES', 'TRUE', 'Y', '1')
      AND cb.recall_charge_rs IS NOT NULL
      AND cmf.manufacturer_id IS NOT NULL
),

-- ── PRN billed quantity ─────────────────────────────────────────────────────
prn AS (
    SELECT
        DATE(p.pos_timestamp + interval '330' minute) AS prn_date,
        p1.variant_id,
        SUM(p1.quantity) AS prn_qty
    FROM pos.pos_invoice p
    INNER JOIN pos.pos_invoice_product_details p1
        ON p1.invoice_id = p.id
       AND p1.lake_active_record = TRUE
       AND p1.insert_ds_ist >= '{{start_date}}'
       AND p1.insert_ds_ist <= CAST(DATE '{{end_date}}' + interval '1' day AS VARCHAR)
    INNER JOIN vendor_entity ve
        ON ve.vendor_id = p.source_entity_vendor_id
       AND ve.is_bcpl = 1
    WHERE p.invoice_type_id = 6
      AND p.lake_active_record = TRUE
      AND p.pos_timestamp >= TIMESTAMP '{{start_date}}' - interval '330' minute
      AND p.pos_timestamp <  TIMESTAMP '{{end_date}}' - interval '330' minute + interval '1' day
      AND p.insert_ds_ist >= '{{start_date}}'
      AND p.insert_ds_ist <= CAST(DATE '{{end_date}}' + interval '1' day AS VARCHAR)
    GROUP BY 1, 2
),
prn_mfr AS (
    SELECT
        prn.prn_date, prn.variant_id, prn.prn_qty,
        pr.item_id, pr.item_name,
        COALESCE(pr.manufacturer_id, im.manufacturer_id) AS manufacturer_id,
        COALESCE(pr.manufacturer,    im.manufacturer)    AS manufacturer
    FROM prn
    LEFT JOIN prod     pr ON pr.variant_id = prn.variant_id
    LEFT JOIN item_mfr im ON im.variant_id = prn.variant_id
),
-- one clause per PRN line: in force on prn_date, most recent contract wins (#8, #13)
priced AS (
    SELECT
        pm.prn_date, pm.variant_id, pm.prn_qty, pm.item_id, pm.item_name,
        pm.manufacturer_id, pm.manufacturer,
        c.contract_id, c.recall_charge_rs,
        ROW_NUMBER() OVER (
            PARTITION BY pm.prn_date, pm.variant_id
            ORDER BY c.contract_updated_at DESC, c.contract_id DESC
        ) AS rn
    FROM prn_mfr pm
    LEFT JOIN clause c
        ON c.manufacturer_id = pm.manufacturer_id
       AND pm.prn_date >= c.effective_date
       AND pm.prn_date <= COALESCE(c.end_date, DATE '9999-12-31')
)

SELECT
    CASE WHEN '{{day_week_month}}' = 'week'  THEN DATE(DATE_TRUNC('week',  p.prn_date))
         WHEN '{{day_week_month}}' = 'month' THEN DATE(DATE_TRUNC('month', p.prn_date))
         ELSE p.prn_date END                        AS duration,
    p.manufacturer_id,
    p.manufacturer,
    CAST(p.item_id AS VARCHAR)                      AS item_id,
    p.item_name,
    MAX(p.contract_id)                              AS contract_id,
    MAX(p.recall_charge_rs)                         AS recall_charge_rs,
    CAST(SUM(p.prn_qty) AS BIGINT)                  AS prn_qty,
    CAST(SUM(CASE WHEN p.recall_charge_rs IS NOT NULL
                  THEN p.prn_qty ELSE 0 END) AS BIGINT) AS chargeable_prn_qty,
    -- DOUBLE, not BIGINT: keep the paise (#11)
    ROUND(COALESCE(SUM(p.prn_qty * p.recall_charge_rs), 0.0), 2) AS recall_assistance_fee,
    MAX(COALESCE(rtv.dmg_rtv_flag, 0))              AS dmg_rtv_flag,
    MAX(COALESCE(rtv.nte_rtv_flag, 0))              AS nte_rtv_flag,
    MAX(CASE WHEN COALESCE(rtv.dmg_rtv_flag, 0) = 1
               OR COALESCE(rtv.nte_rtv_flag, 0) = 1 THEN 1 ELSE 0 END) AS rtv_eligible
FROM priced p
LEFT JOIN rtv ON rtv.item_id = p.item_id
WHERE p.rn = 1
GROUP BY 1, 2, 3, 4, 5
'''

print(f'{len(RECALL_FEE_SQL):,} chars')

In [ ]:
# =============================================================================
# RUN — batched by date window, concatenated, written once
# =============================================================================
FEE_COLUMNS = [
    'duration', 'manufacturer_id', 'manufacturer', 'item_id', 'item_name',
    'contract_id', 'recall_charge_rs', 'prn_qty', 'chargeable_prn_qty',
    'recall_assistance_fee', 'dmg_rtv_flag', 'nte_rtv_flag', 'rtv_eligible',
]

if ckpt_ok(FEE_PATH):
    log(f'checkpoint exists at {FEE_PATH} - delete the file to re-extract',
        label='recall_assistance_fee')
    fee_df = pd.read_parquet(FEE_PATH)
else:
    windows = list(date_windows(START_DATE, END_DATE, BATCH_DAYS))
    log(f'{len(windows)} window(s) of up to {BATCH_DAYS}d', label='recall_assistance_fee')

    frames = []
    for i, (w_start, w_end) in enumerate(windows, 1):
        part = sql(
            RECALL_FEE_SQL,
            label=f'recall_assistance_fee {i}/{len(windows)} {w_start}..{w_end}',
            start_date=w_start,
            end_date=w_end,
            day_week_month=DAY_WEEK_MONTH,
            entity_token=ENTITY_TOKEN,
        )
        frames.append(part)

    fee_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=FEE_COLUMNS)

    # A month/week grain straddles window boundaries, so re-aggregate after concat.
    if len(fee_df) and DAY_WEEK_MONTH in ('week', 'month'):
        before = len(fee_df)
        fee_df = (fee_df
                  .groupby(['duration', 'manufacturer_id', 'manufacturer', 'item_id', 'item_name'],
                           dropna=False, as_index=False)
                  .agg(contract_id=('contract_id', 'max'),
                       recall_charge_rs=('recall_charge_rs', 'max'),
                       prn_qty=('prn_qty', 'sum'),
                       chargeable_prn_qty=('chargeable_prn_qty', 'sum'),
                       recall_assistance_fee=('recall_assistance_fee', 'sum'),
                       dmg_rtv_flag=('dmg_rtv_flag', 'max'),
                       nte_rtv_flag=('nte_rtv_flag', 'max'),
                       rtv_eligible=('rtv_eligible', 'max')))
        log(f'cross-window re-aggregation: {before:,} -> {len(fee_df):,} rows',
            label='recall_assistance_fee')

    fee_df = fee_df.reindex(columns=FEE_COLUMNS)
    fee_df.to_parquet(FEE_PATH, index=False, compression='snappy')
    log(f'wrote {FEE_PATH}  ({os.path.getsize(FEE_PATH)/1e6:.1f} MB)',
        label='recall_assistance_fee')

print(f'\n{len(fee_df):,} rows x {fee_df.shape[1]} cols')
fee_df.head(20)

In [ ]:
# =============================================================================
# RECONCILE — does the output hold together?
# =============================================================================
if not len(fee_df):
    print('EMPTY RESULT. Work through the probes above before anything else:')
    print('  - Probe 2 empty  -> ENTITY_TOKEN matches no legal_name')
    print('  - Probe 3b empty -> no contract carries the recall clause in this window')
    print('  - both non-empty -> suspect the pos_invoice date window / invoice_type_id')
else:
    total_qty = fee_df.prn_qty.sum()
    charge_qty = fee_df.chargeable_prn_qty.sum()
    total_fee = fee_df.recall_assistance_fee.sum()

    print(f'rows                       {len(fee_df):>15,}')
    print(f'distinct items             {fee_df.item_id.nunique():>15,}')
    print(f'distinct manufacturers     {fee_df.manufacturer_id.nunique():>15,}')
    print(f'PRN qty (all)              {total_qty:>15,}')
    print(f'PRN qty (has a clause)     {charge_qty:>15,}   {charge_qty/total_qty:>6.1%}')
    print(f'recall assistance fee      {total_fee:>15,.2f}')
    print(f'duration range             {fee_df.duration.min()} -> {fee_df.duration.max()}')

    # Identity check: fee must equal chargeable qty x charge, to rounding.
    priced_rows = fee_df[fee_df.recall_charge_rs.notna()]
    implied = (priced_rows.chargeable_prn_qty * priced_rows.recall_charge_rs).sum()
    drift = abs(implied - total_fee)
    print(f'\nqty x charge vs reported fee   drift = {drift:,.2f}', end='  ')
    print('OK' if drift < 1.0 else 'MISMATCH - investigate before using this output')

    # Grain check: one row per (duration, item, manufacturer). A dupe here means a
    # join fanned out somewhere and the quantities are inflated.
    key = ['duration', 'item_id', 'manufacturer_id']
    dupes = fee_df[fee_df.duplicated(key, keep=False)]
    print(f'duplicate grain keys           {len(dupes):,}', end='  ')
    print('OK' if not len(dupes) else 'FAN-OUT - quantities are inflated')

    # Coverage: unpriced returns are expected (no contract), but a near-zero
    # chargeable share usually means the manufacturer_id join is not landing.
    if total_qty and charge_qty / total_qty < 0.01:
        print('\nWARNING: <1% of PRN qty matched a clause. Check that '
              'clause.manufacturer_id and prod.manufacturer_id are the same id space '
              '(contract mapped_id vs rpc.product_manufacturer.id).')

    print('\nTop 15 manufacturers by fee:')
    print(fee_df.groupby(['manufacturer_id', 'manufacturer'], dropna=False)
                .agg(prn_qty=('prn_qty', 'sum'),
                     fee=('recall_assistance_fee', 'sum'))
                .sort_values('fee', ascending=False)
                .head(15).to_string())

## Known limits

- **`manufacturer_id` id-space.** The contract side yields
  `clm_contract_mapping.mapped_id`; the item side yields
  `rpc.product_manufacturer.id` (via brand), with `ba_etls.item_details.manufacturer_id`
  as fallback. The repo joins these as `VARCHAR` on the assumption they are the same
  id space. The reconcile cell warns if the chargeable share collapses, which is what
  a mismatch would look like. Confirm on a known manufacturer before signing off.
- **Multi-value clause answers.** If one contract carries several
  `recall_charges_rs_v1` values, `MAX` takes the largest. That is at least numerically
  correct now, but it is still a silent choice — check Probe 3b for contracts with
  more than one charge.
- **Unpriced returns are retained** with `recall_charge_rs = NULL` and
  `recall_assistance_fee = 0`, so `prn_qty` reconciles against a plain PRN report.
  Filter on `recall_charge_rs.notna()` for the billable subset only.
- **`return_type` is not filtered.** `IOCC-Contracts (1).ipynb` cell 14 restricts the
  return-policy CTE to `return_type = 2`; the Word doc version does not. This notebook
  follows the Word doc and keeps all return types. If `return_type = 2` is the correct
  restriction, add it to all three `rtv_src` branches.